In [0]:
yellow_landing = "abfss://landing@stadbenterprisede98.dfs.core.windows.net/yellow"
fhv_landing = "abfss://landing@stadbenterprisede98.dfs.core.windows.net/fhv"
fhvhv_landing = "abfss://landing@stadbenterprisede98.dfs.core.windows.net/fhvhv"

yellow_bronze = "abfss://bronze@stadbenterprisede98.dfs.core.windows.net/yellow"
fhv_bronze = "abfss://bronze@stadbenterprisede98.dfs.core.windows.net/fhv"
fhvhv_bronze = "abfss://bronze@stadbenterprisede98.dfs.core.windows.net/fhvhv"

yellow_checkpoint = "abfss://checkpoints@stadbenterprisede98.dfs.core.windows.net/bronze/yellow"
fhv_checkpoint = "abfss://checkpoints@stadbenterprisede98.dfs.core.windows.net/bronze/fhv"
fhvhv_checkpoint = "abfss://checkpoints@stadbenterprisede98.dfs.core.windows.net/bronze/fhvhv"

In [0]:
yellow_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.useManagedFileEvents", "true")
    .option("cloudFiles.schemaLocation", yellow_checkpoint + "/schema")
    .load(yellow_landing)
)

In [0]:
yellow_query = (
    yellow_stream.writeStream
    .format("delta")
    .option("checkpointLocation", yellow_checkpoint + "/state")
    .trigger(availableNow=True)
    .start(yellow_bronze)
)

yellow_query.awaitTermination()

In [0]:
dbutils.fs.ls(yellow_bronze)

In [0]:
fhv_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.useManagedFileEvents", "true")
    .option("cloudFiles.schemaLocation", fhv_checkpoint + "/schema")
    .load(fhv_landing)
)

In [0]:
fhv_query = (
    fhv_stream.writeStream
    .format("delta")
    .option("checkpointLocation", fhv_checkpoint + "/state")
    .trigger(availableNow=True)
    .start(fhv_bronze)
)

fhv_query.awaitTermination()

In [0]:
dbutils.fs.ls(fhv_bronze)

In [0]:
if len(dbutils.fs.ls(fhvhv_landing)) > 0:

    fhvhv_stream = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.useManagedFileEvents", "true")
        .option("cloudFiles.schemaLocation", fhvhv_checkpoint + "/schema")
        .load(fhvhv_landing)
    )

    fhvhv_query = (
        fhvhv_stream.writeStream
        .format("delta")
        .option("checkpointLocation", fhvhv_checkpoint + "/state")
        .trigger(availableNow=True)
        .start(fhvhv_bronze)
    )

    fhvhv_query.awaitTermination()

    spark.sql("""
        CREATE TABLE IF NOT EXISTS enterprise_de.bronze.fhvhv_trips_raw
        USING DELTA
        LOCATION 'abfss://bronze@stadbenterprisede98.dfs.core.windows.net/fhvhv'
    """)

    print("HVFHV Bronze ingestion completed and table registered.")

else:
    print("No HVFHV files in landing yet — skipping.")

In [0]:
print("YELLOW BRONZE:")
display(dbutils.fs.ls(yellow_bronze))

print("FHV BRONZE:")
display(dbutils.fs.ls(fhv_bronze))

if len(dbutils.fs.ls(fhvhv_landing)) > 0:
    print("HVFHV BRONZE:")
    display(dbutils.fs.ls(fhvhv_bronze))
else:
    print("HVFHV has not started yet.")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS enterprise_de.bronze.yellow_trips_raw
USING DELTA
LOCATION 'abfss://bronze@stadbenterprisede98.dfs.core.windows.net/yellow';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS enterprise_de.bronze.fhv_trips_raw
USING DELTA
LOCATION 'abfss://bronze@stadbenterprisede98.dfs.core.windows.net/fhv';

In [0]:
%sql
SHOW TABLES IN enterprise_de.bronze;

In [0]:
if spark.catalog.tableExists("enterprise_de.bronze.fhvhv_trips_raw"):

    counts = spark.sql("""
        SELECT 'yellow' AS source, COUNT(*) AS rows
        FROM enterprise_de.bronze.yellow_trips_raw

        UNION ALL

        SELECT 'fhv' AS source, COUNT(*) AS rows
        FROM enterprise_de.bronze.fhv_trips_raw

        UNION ALL

        SELECT 'fhvhv' AS source, COUNT(*) AS rows
        FROM enterprise_de.bronze.fhvhv_trips_raw
    """)
    display(counts)

else:

    counts = spark.sql("""
        SELECT 'yellow' AS source, COUNT(*) AS rows
        FROM enterprise_de.bronze.yellow_trips_raw

        UNION ALL

        SELECT 'fhv' AS source, COUNT(*) AS rows
        FROM enterprise_de.bronze.fhv_trips_raw
    """)

display(counts)
display("fhvhv table is not found yet")